This notebook shows the diffs of Wan2.2 and Wan2.2 + Prompt Relay

In [ ]:
# generate.py

"""
Implementation would be almost a 1:1 copy-paste. 
"""

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------
# id: modification-1
# status: Done
def _parse_args():
    parser = argparse.ArgumentParser(
        description="Generate a image or video from a text prompt or image using Wan"
    )

    ########## Prompt Relay Prompt Input ##########
    # add a argument to accept a JSON configuration file instead of a standard string prompt.
    parser.add_argument(
        "--prompt_filepath",
        type=str,
        default=None,
        help="The file of the input prompts containing the timesteps for each prompt."
    )
    ################################################## 

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
if args.prompt is None:
        args.prompt = EXAMPLE_PROMPT[args.task]["prompt"]

# New
# id: modification-2
# status: Done
if args.prompt is None:
         ########## Prompt Relay (Replace prompt if prompt_filepath json is provided) ########## 
        if args.prompt_filepath is not None:
            import json
            with open(args.prompt_filepath, 'r') as f:
                prompt_data = json.load(f)
            full_prompt = prompt_data.get("global_prompt", "")
            local_prompts = prompt_data.get("local_prompts", [])
            args.prompt = full_prompt + " ".join(local_prompts)
        else:
            args.prompt = EXAMPLE_PROMPT[args.task]["prompt"]

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# id: modification-3.1, 3.2.0, 3.2.1, 3.2.2, 3.2.3
# status: Done
def generate(args):
    
    # ... other code ...
    
    video = wan_t2v.generate(
                args.prompt,
                size=SIZE_CONFIGS[args.size],
                frame_num=args.frame_num,
                shift=args.sample_shift,
                sample_solver=args.sample_solver,
                sampling_steps=args.sample_steps,
                guide_scale=args.sample_guide_scale,
                seed=args.base_seed,
                offload_model=args.offload_model,

                # pass the raw JSON file path down into wan_t2v.generate()
                # if no JSON provided, the model must be able to work normally as if there is no prompt-relay
                prompt_filepath = args.prompt_filepath)

In [ ]:
# wan/text2video.py

"""
CogVideoX has its own VAE with its own temporal compression ratio. Need to write a similar _prepare_prompts function that calculates latent_frames based on CogVideoX's specific vae_stride (or patch size equivalent).

Once we build that payload (q_token_idx), we can just attach it to whatever CogVideoX uses as its conditioning dictionary (often encoder_hidden_states or a kwargs dict in PyTorch/Diffusers) so it travels down into the DiT block.
"""

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
# (Method does not exist in the baseline model)

# New
# id: modification-5
# status: Done
    ########## Prompt Relay  ########## 
    # Helper method: Calculates how many latent frames belong to each segment, 
    # finds the exact T5 token indices for each local prompt, and packages 
    # this into q_token_idx
    def _prepare_prompts(self, global_prompt, local_prompts, segment_lengths, frame_num, size):
        
        tokenizer = self.text_encoder.tokenizer

        # Calculate how time and space is compressed by the VAE
        latent_frames = (frame_num - 1) // self.vae_stride[0] + 1
        width, height = size
        h_lat = int(height) // self.vae_stride[1]
        w_lat = int(width) // self.vae_stride[2]
        h_patches = h_lat // self.patch_size[1]
        w_patches = w_lat // self.patch_size[2]
        tokens_per_frame = int(h_patches) * int(w_patches)

        full_prompt = global_prompt + "".join(local_prompts)
        
        full_ids = tokenizer(full_prompt, add_special_tokens=True, padding=False, return_mask=False)[0].tolist()
        
        # Find exact text token boundaries for each sub-prompt
        def sentence_to_token_indices(subsentences):
            def find_subsequence(haystack, needle):
                for start in range(len(haystack) - len(needle) + 1):
                    if haystack[start: start + len(needle)] == needle: 
                        return (start, start + len(needle))
                
            token_indices = {}
            for subsentence in subsentences:
                sub_ids = tokenizer(subsentence, padding=False, add_special_tokens=False)[0].tolist()
                match = find_subsequence(full_ids, sub_ids)
                if match is None:
                    raise ValueError(f"Subsentence not found in full prompt: {subsentence}")
                
                token_indices[subsentence] = match
            return token_indices

        # Build the payload containing the temporal windows and Gaussian decay sigma
        def build_q_token_idx(frame_intervals, token_spans, tokens_per_frame):        
            q_token_idx = []
            epsilon = 1e-3

            if len(frame_intervals)!=0:
                for _, (frame_start, frame_end, subsentences) in enumerate(frame_intervals): 
                    spans = []
                    for subsentence in subsentences:
                        start, end = token_spans[subsentence]
                        spans.extend(range(start, end))

                    window = (frame_end - frame_start)//2 - 2 
                    sigma =  0.1448 
                    
                    payload = {
                        "window": window,
                        "sigma": torch.tensor(sigma, dtype=torch.float16),
                        "midpoint": (frame_start + frame_end) // 2,
                        "tokens_per_frame": tokens_per_frame,
                        "local_token_idx": torch.tensor(spans, dtype=torch.long),
                    }
                    
                    q_token_idx.append(payload)
            return q_token_idx

        # Map the requested segment lengths to the compressed latent frames
        spans = sentence_to_token_indices(local_prompts)
        
        if len(local_prompts) != 0:
            step = math.ceil(latent_frames / len(local_prompts)) 

            if len(segment_lengths) != 0:
                frame_intervals = []
                frame_counter = 0
                for i, seg_len in enumerate(segment_lengths):
                    frame_start = frame_counter
                    frame_end = min(frame_counter + seg_len, latent_frames)
                    frame_intervals.append((frame_start, frame_end, [local_prompts[i]]))
                    frame_counter += seg_len
            else:
                frame_intervals = [(step*i, 
                                    min(step*(i+1), latent_frames), 
                                    [local_prompts[i]]) for i in range(len(local_prompts))]
                
            q_token_idx = build_q_token_idx(frame_intervals = frame_intervals,
                                            token_spans = spans,
                                            tokens_per_frame = tokens_per_frame,
                                            )
        else:
            q_token_idx = None
        return q_token_idx, full_prompt


# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
    def generate(self,
                 input_prompt,
                 # ... other args ...
                 offload_model=True):

# New
# id: modification-4
# status: Done
    def generate(self,
                 input_prompt,
                 # ... other args ...
                 offload_model=True,
                 prompt_filepath=None): # add the new argument to the generate signature

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
        # preprocess
        guide_scale = (guide_scale, guide_scale) if isinstance(

# New
# id: modification-6
# status: Done
        # If the JSON is provided, load the segments and trigger _prepare_prompts
        if prompt_filepath is not None:
            ########## Prompt Relay  ########## 
            with open(prompt_filepath, 'r') as f:
                prompts = json.load(f)
        
            global_prompt = prompts.get("global_prompt", "")
            local_prompts = prompts.get("local_prompts", [])
            segment_lengths = prompts.get("segment_lengths", [])
            
            # Pad with a space so tokenization doesn't fuse words across boundaries
            local_prompts = [" " + lp for lp in local_prompts]

            cross_attn_q_token_idx, input_prompt = self._prepare_prompts(global_prompt, local_prompts, segment_lengths, frame_num, size)


        # preprocess
        guide_scale = (guide_scale, guide_scale) if isinstance(

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
            # sample videos
            latents = noise

            arg_c = {'context': context, 'seq_len': seq_len}
            arg_null = {'context': context_null, 'seq_len': seq_len}

# New
# id: modification-7
# status: Done
            # sample videos
            latents = noise

            # Pack the calculated token boundaries into the conditioning dictionary (arg_c)
            # This is passed into the model() forward pass at every timestep
            arg_c = {
                'context': context,
                'seq_len': seq_len,
                'cross_attn_q_token_idx': None if prompt_filepath is None else cross_attn_q_token_idx
            }
            arg_null = {'context': context_null, 'seq_len': seq_len}

In [ ]:
# wan/distributed/sequence_parallel.py

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
def sp_dit_forward(
    self,
    x,
    t,
    context,
    seq_len,
    y=None,
):

# New
# id: modification-8
# status: Done
def sp_dit_forward(
    self,
    x,
    t,
    context,
    seq_len,
    y=None,
    cross_attn_q_token_idx=None,
    self_attention_map=None,
):


# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
    # arguments
    kwargs = dict(
        e=e0,
        seq_lens=seq_lens,
        grid_sizes=grid_sizes,
        freqs=self.freqs,
        context=context,
        context_lens=context_lens)

# New
# id: modification-8.1 modification 8.2 modification-9
# status: Done
    # arguments
    kwargs = dict(
        e=e0,
        seq_lens=seq_lens,
        grid_sizes=grid_sizes,
        freqs=self.freqs,
        context=context,
        context_lens=context_lens,
        cross_attn_q_token_idx=cross_attn_q_token_idx,
        self_attention_map=self_attention_map,
    )

# NOTE: self_attention_map for CogVideoX has been skipped rn because it appears that code changes related to it in Wan2.2 are commented out right now

In [ ]:
# wan/modules/model.py

"""
Maps the structural changes in the model block to pass routing indices (cross_attn_q_token_idx) down the sequence.
Introduces custom chunked attention functions to compute temporal cost maps for prompt relay segmentation.
"""

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# New
# id: modification-
# status: 
import torch.nn.functional as F

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
# (Methods do not exist in the baseline model)

# New
# id: modification-11
# status: Done
# Note: CogVideoX's build_temporal_cost requires the text_seq_length offset because it uses Joint Attention, unlike Wan2.2's isolated cross-attention.
########## Prompt Relay  ########## 
def build_temporal_cost(q_token_idx, Lq, Lk, device, dtype):
    #Assume that each segment is equal in length
    # q_token_idx.sort(key = lambda x: x ['midpoint'])
    offset = torch.zeros(Lq, Lk, device=device, dtype=dtype)

    #The frame number for each query token
    tokens_per_frame = int(q_token_idx[0]['tokens_per_frame'])
    query_frames = (
        torch.arange(Lq, device=device, dtype=torch.long)
        // tokens_per_frame
    )

    for seg in q_token_idx:
        w = seg['window']
        sigma = torch.tensor(seg['sigma'], dtype=torch.float32, device=device)
        local = seg['local_token_idx'].to(device=device)
        midpoint = torch.tensor(seg['midpoint'], dtype=torch.float32, device=device)

        d = (query_frames.float()[:, None] - midpoint).abs()
        cost = (torch.relu(d - w) ** 2) / (2 * sigma ** 2)

        offset[:, local] = cost.to(offset.dtype)
        
    del query_frames, sigma
    return offset

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# id: modification-12
# status: done
# Note: CogVideoX's chunked_softmax_attention requires the text_seq_length parameter and a float32 datatype cast to prevent matrix multiplication crashes.
def chunked_softmax_attention(q, k, v, q_token_idx, chunk_size=16):
    q = q.transpose(1,2)
    k = k.transpose(1,2)
    v = v.transpose(1,2)

    B, H, Lq, D = q.shape
    _, _, Lk, _ = k.shape
    scale = 1.0 / math.sqrt(D)

    temporal_cost_map = build_temporal_cost(q_token_idx, Lq, Lk, q.device, q.dtype)
    out = torch.zeros(B, H, Lq, D, device=q.device, dtype=q.dtype)

    for start in range(0, Lq, chunk_size):
        end = min(start + chunk_size, Lq)
        logits = torch.matmul(q[:, :, start:end, :], k.transpose(-2, -1)) * scale 

        mask_chunk = temporal_cost_map[start:end].unsqueeze(0).unsqueeze(0)
        logits = logits - mask_chunk.float() 
        attn = torch.softmax(logits, dim=-1)
        out[:, :, start:end] = torch.matmul(attn, v)
        
        del logits, attn
        
    return out.transpose(1,2)

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
def rope_apply(x, grid_sizes, freqs):
    # ... previous code ...
    for i, (f, h, w) in enumerate(grid_sizes.tolist()):
        seq_len = f * h * w

        # precompute multipliers
        x_i = torch.view_as_complex(x[i, :seq_len].to(torch.float64).reshape(
            seq_len, n, -1, 2))
        
        # precompute multipliers
        x_i = torch.view_as_complex(x_visual.to(torch.float64).reshape(
            seq_len, n, -1, 2))
        freqs_i = torch.cat([
            freqs[0][:f].view(f, 1, 1, -1).expand(f, h, w, -1),
            freqs[1][:h].view(1, h, 1, -1).expand(f, h, w, -1),
            freqs[2][:w].view(1, 1, w, -1).expand(f, h, w, -1)
        ],
                            dim=-1).reshape(seq_len, 1, -1)

        # apply rotary embedding
        x_i = torch.view_as_real(x_i * freqs_i).flatten(2)
        x_i = torch.cat([x_i, x[i, seq_len:]])

        # append to collection
        output.append(x_i)
    return torch.stack(output).float()

# New
# id: modification-
# status: 
def rope_apply(x, grid_sizes, freqs):
    # ... previous code ...
    for i, (f, h, w) in enumerate(grid_sizes.tolist()):
        seq_len = f * h * w

        x_visual = x[i, :seq_len].clone()
        x_extra = x[i, seq_len:].clone()
        
        # precompute multipliers
        x_i = torch.view_as_complex(x_visual.to(torch.float64).reshape(
            seq_len, n, -1, 2))
        
        # precompute multipliers
        x_i = torch.view_as_complex(x_visual.to(torch.float64).reshape(
            seq_len, n, -1, 2))
        freqs_i = torch.cat([
            freqs[0][:f].view(f, 1, 1, -1).expand(f, h, w, -1),
            freqs[1][:h].view(1, h, 1, -1).expand(f, h, w, -1),
            freqs[2][:w].view(1, 1, w, -1).expand(f, h, w, -1)
        ],
                            dim=-1).reshape(seq_len, 1, -1)

        # apply rotary embedding
        x_visual = torch.view_as_real(x_i * freqs_i).flatten(2)
        x_i = torch.cat([x_visual, x_extra])

        # append to collection
        output.append(x_i)
    return torch.stack(output).float()

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
class WanSelfAttention(nn.Module):
    # ... Code for init ...

    def forward(self, x, seq_lens, grid_sizes, freqs):
        # ...
        q, k, v = qkv_fn(x)

        x = flash_attention(
            q=rope_apply(q, grid_sizes, freqs),
            k=rope_apply(k, grid_sizes, freqs),
            v=v,
            k_lens=seq_lens,
            window_size=self.window_size)

        # output
        x = x.flatten(2)
        x = self.o(x)
        return x

# New
# id: modification-nil
# status: Not implemented
# Note: These do not map to CogVideoX modifications. 
#       The rope_apply changes were unique to Wan's specific rotary embedding implementation, and the self_attention_map was tied to a spatial control feature that we explicitly noted was skipped for this implementation as code was commented.
class WanSelfAttention(nn.Module):
    # ... Code for init ...

    def forward(self, x, seq_lens, grid_sizes, freqs, self_attention_map=None):
        # ...
        q, k, v = qkv_fn(x)

        q = rope_apply(q, grid_sizes, freqs)
        k = rope_apply(k, grid_sizes, freqs)

        # if self_attention_map is None:
        x = flash_attention(
            q=q,
            k=k,
            v=v,
            k_lens=seq_lens,
            window_size=self.window_size)

        # output
        x = x.flatten(2)
        x = self.o(x)
        return x

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
class WanCrossAttention(WanSelfAttention):
    def forward(self, x, context, context_lens):
        # ...
        # compute attention
        x = flash_attention(q, k, v, k_lens=context_lens)

# New
# id: modification-14
# status: Done
# Note: Wan2.2 introduced an if not q_token_idx: block to decide whether to use native flash_attention or the custom chunked_softmax_attention. 
#       Mod-14 replicates this logic by placing an if/else wrapper around PyTorch's F.scaled_dot_product_attention.
class WanCrossAttention(WanSelfAttention):
    def forward(self, x, context, context_lens, q_token_idx=None):
        r"""
        Args:
            q_token_idx (list[tuple[int, int, Tensor | list[int]]] | None):
                Optional routing that restricts which context tokens each query range can attend to.
        """
        # ...
        # compute attention
        if not q_token_idx:
            x = flash_attention(q, k, v, k_lens=context_lens)
        else:
            x = chunked_softmax_attention(q,k,v, q_token_idx = q_token_idx)

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
    def forward(
        self, x, e, seq_lens, grid_sizes, freqs, context, context_lens,
    ):
        # ...
        y = self.self_attn(
            self.norm1(x).float() * (1 + e[1].squeeze(2)) + e[0].squeeze(2),
            seq_lens, grid_sizes, freqs)
        # ...
        def cross_attn_ffn(x, context, context_lens, e):
            x = x + self.cross_attn(self.norm3(x), context, context_lens)

# New
# id: modification-10
# status: Done
    def forward(
        self, x, e, seq_lens, grid_sizes, freqs, context, context_lens,
        cross_attn_q_token_idx=None,
        self_attention_map=None,
    ):
        # ...
        y = self.self_attn(
            self.norm1(x).float() * (1 + e[1].squeeze(2)) + e[0].squeeze(2),
            seq_lens, grid_sizes, freqs, self_attention_map=self_attention_map)
        # ...
        def cross_attn_ffn(x, context, context_lens, e):
            x = x + self.cross_attn(self.norm3(x),
                                    context,
                                    context_lens,
                                    q_token_idx=cross_attn_q_token_idx)

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# Original
    def forward(
        self, x, t, context, seq_len, y=None,
    ):
        # ...
        kwargs = dict(
            e=e0,
            seq_lens=seq_lens,
            grid_sizes=grid_sizes,
            freqs=self.freqs,
            context=context,
            context_lens=context_lens)

# New
# id: modification-13
# status: Done
# Note: Wan2.2, the forward function of the cross-attention class was updated to accept q_token_idx=None. Because CogVideoX abstracts attention through processor classes,
#       Mod-13 achieved this goal by adding cross_attn_q_token_idx to the CogVideoXAttnProcessor2_0.__call__ signature.
    def forward(
        self, x, t, context, seq_len, y=None,
        cross_attn_q_token_idx=None,
        self_attention_map=None,
    ):
        # ...
        kwargs = dict(
            e=e0,
            seq_lens=seq_lens,
            grid_sizes=grid_sizes,
            freqs=self.freqs,
            context=context,
            context_lens=context_lens,
            cross_attn_q_token_idx=cross_attn_q_token_idx,
            self_attention_map=self_attention_map,
        )

### Step 1. Input Ingestion (generate.py)
* **Original:** The model accepts a standard string prompt via standard arguments (`args.prompt`).
* **New:** A new `--prompt_filepath` argument is introduced to accept a JSON file. If provided, the script extracts a `global_prompt` and a list of `local_prompts`, concatenates them into a single `full_prompt`, and passes the file path directly into the generation function (`wan_t2v.generate`).

### Step 2. Tokenization & Alignment (wan/text2video.py)
* **Original:** Text is tokenized standardly, and embeddings are passed into the model conditioning dictionary.
* **New:** The script introduces a `_prepare_prompts` function to spatially and temporally map the text. 
    * It calculates the video's compressed dimensions (`latent_frames`, `h_patches`, `w_patches`) based on the VAE stride.
    * It tokenizes the `full_prompt` and finds the exact starting and ending token indices for each `local_prompt`.
    * It builds `cross_attn_q_token_idx`, a list of dictionaries containing the temporal `window`, Gaussian `sigma`, frame `midpoint`, and `local_token_idx` (allowed text tokens) for each specific segment.
    * This payload is injected into the conditioning dictionary (`arg_c`) to be passed into the model's forward pass at every timestep.

### Step 3. Routing the Payload (wan/distributed/sequence_parallel.py & wan/modules/model.py)
* **Original:** Only standard conditioning (`e`, `context`, `seq_len`) is routed through the transformer blocks.
* **New:** The `sp_dit_forward` and `WanModel.forward` functions are updated to accept `cross_attn_q_token_idx`. This payload is passed into the `kwargs` dictionary, traversing down through `WanAttentionBlock` and ultimately arriving at `WanCrossAttention`.

### Step 4. Masked Cross-Attention (wan/modules/model.py)
* **Original:** `WanCrossAttention` computes query/key/values and passes them directly into a standard `flash_attention` block.
* **New:** `WanCrossAttention` intercepts the flow. 
    * If `q_token_idx` is present, it bypasses `flash_attention` and routes to a custom `chunked_softmax_attention`.
    * The `build_temporal_cost` function unpacks the payload to generate a mathematical penalty (a Gaussian decay mask).
    * This mask artificially lowers the attention logits for specific text tokens outside their assigned temporal windows before the softmax is applied, forcing specific frames to only "see" their designated prompt segments.